# Anime Face Generation: VAE + DDPM from scratch

Run each stage as a script (`!python x.py`), same commands you'd use in a terminal. No PyTorch Lightning - plain PyTorch training loops so everything is easy to read/debug.

**Runtime:** GPU (T4). In Colab: `Runtime > Change runtime type > T4 GPU`. On Kaggle: enable GPU T4 x2 in the notebook settings sidebar.

## 1. Setup: clone/upload project + install deps

In [ ]:
# If you uploaded this project as a zip/folder to Colab, cd into it.
# If instead you're pulling from GitHub, uncomment and edit the clone line:
# !git clone https://github.com/<you>/<repo>.git
# %cd <repo>

!pip install -q -r requirements.txt


## 2. Kaggle API token

`data.py` uses `kagglehub` to download the dataset, which needs a Kaggle API token.

- **On Kaggle notebooks**: already configured, skip this cell.
- **On Colab**: upload your `kaggle.json` (from https://www.kaggle.com/settings -> API -> Create New Token) using the file upload cell below, or set the two env vars directly.

In [ ]:
import os

# Option A: upload kaggle.json via Colab's file picker, then run this cell
# from google.colab import files
# uploaded = files.upload()  # select kaggle.json
# os.makedirs('/root/.kaggle', exist_ok=True)
# !mv kaggle.json /root/.kaggle/kaggle.json
# !chmod 600 /root/.kaggle/kaggle.json

# Option B: set credentials directly (replace with your own)
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'


## 3. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))


## 4. Download + preprocess data

Downloads the anime face dataset (~63.6k images) via `kagglehub`, resizes to 64x64, and splits into `data/train/all/` and `data/test/all/`. Only needs to run once (re-run is a no-op if `data/train` is already populated).

In [ ]:
!python data.py

## 5. Train the VAE

Watch both `recon` and `kl` loss terms printed every log step / epoch average - see the README for what healthy vs. unhealthy curves look like. Checkpoints save every epoch to `checkpoints/vae.pt` (resumable via `--resume checkpoints/vae.pt` if your session gets cut off).

In [ ]:
!python train_vae.py

### Preview VAE samples/reconstructions so far

In [ ]:
from PIL import Image
import glob

latest = sorted(glob.glob('samples/vae/epoch_*.png'))[-1]
print(latest)
Image.open(latest)


## 6. Evaluate the VAE

Generates a preview grid + a latent interpolation, and dumps `--num_samples` generated images to `samples/vae_eval/` for FID/IS scoring later. Lower `--num_samples` here if you're short on time/quota.

In [ ]:
!python eval_vae.py --interpolate --num_samples 2000

In [ ]:
from PIL import Image
Image.open('samples/vae_eval/preview_grid.png')


## 7. Train the DDPM

Judge convergence by the **epoch average** `avg_mse_loss`, not individual steps (those are noisy since each step samples a random diffusion timestep). See the README for expected loss ranges. Checkpoints (including EMA weights, used for sampling) save every epoch to `checkpoints/ddpm.pt` and are resumable the same way as the VAE.

In [ ]:
!python train_ddpm.py

### Preview DDPM samples so far

In [ ]:
from PIL import Image
import glob

latest = sorted(glob.glob('samples/ddpm/epoch_*.png'))[-1]
print(latest)
Image.open(latest)


## 8. Evaluate the DDPM

DDPM ancestral sampling is inherently slow (1000 sequential steps per batch) - this will take noticeably longer than the VAE eval. Reduce `--num_samples` if you're short on quota.

In [ ]:
!python eval_ddpm.py --num_samples 2000

In [ ]:
from PIL import Image
Image.open('samples/ddpm_eval/preview_grid.png')


## 9. Benchmark: FID + Inception Score

Scores both models' generated images against the real held-out test split.

In [ ]:
!python benchmark.py --both

## 10. Push results to GitHub

Configure git, then commit `checkpoints/`, `samples/`, and `logs/` alongside the code. Consider Git LFS for the `.pt` checkpoint files if they're large.

In [ ]:
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"
# !git add -A
# !git commit -m "Train VAE + DDPM on anime faces, add results"
# !git push
